In [1]:
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()
import sys
sys.path.append("../")
from dotenv import load_dotenv
load_dotenv()

from src.llm_utils import LLM_wrapper, RepoAssessment
from src.llm_utils import REPO_ASSESSMENT_PROMPT as repo_assessment_prompt
from src.repo_utils import RepoStatus

In [2]:
MODEL_NAME = "gpt-5.2-2025-12-11"
CONTEXT_WINDOW = 395000 # gpt 5.2's context window is 400k tokens

In [3]:
llm_wrapper = LLM_wrapper(
    model_name=MODEL_NAME,
    system_prompt=repo_assessment_prompt,
    output_format_class=RepoAssessment
)

In [4]:
df_pred = llm_wrapper.assess_dataframe(
    input_file_path="../data/03_repo_assessment/gt_repo_assessment.csv",
    text_column="repo_content",
    output_dir="../data/03_repo_assessment/",
    # we only want to process rows that were properly cloned
    row_filter=lambda row: row["repo_status"] == RepoStatus.OK
)

Total rows: 35, Already processed: 0, Remaining (after filter): 29


100%|██████████| 29/29 [03:33<00:00,  7.35s/it]

Processing complete. Final results saved.


In [5]:
df_pred

,PMCID,repo_url,gt_is_empty,gt_contains_readme,gt_readme_purpose_and_outputs,gt_contains_requirements,gt_requirements_dependency_versions,gt_contains_license,gt_sufficient_code_documentation,gt_is_modular_and_structured,...,sufficient_code_documentation,is_modular_and_structured,implements_tests,fixes_seed_if_stochastic,lists_hardware_requirements,contains_link_to_paper,contains_citation,includes_data_or_sample,comments_and_explanations,coding_languages
0,PMC11307559,https://seakheeoh76.shinyapps.io/XGBoost_BA_pr...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,PMC12121029,https://github.com/EnzhaoZhu/Common-risks-pred...,False,True,True,True,True,False,False,True,...,True,True,False,None,True,False,False,False,Repository contains a small set of PyTorch/Tra...,[python]
2,PMC11701130,https://github.com/CalvinSMU/AiLES,True,True,False,False,NaN,False,False,False,...,False,False,False,None,False,True,True,False,Repository content is effectively just a READM...,None
3,PMC8175333,https://github.com/ATARIQ5,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,PMC8932490,https://github.com/Aneja-Lab-Yale/Aneja-Lab-Pu...,False,True,True,True,True,False,True,True,...,True,True,False,False,False,True,False,False,"Strengths: clear README with purpose, paper li...",[python]
5,PMC11040883,https://github.com/laamit/champCalculator,False,True,True,True,True,True,False,True,...,True,True,True,None,False,False,False,True,Well-structured R package + Shiny app built wi...,"[r, dockerfile]"
6,PMC10545794,https://github.com/TheDecodeLab/Prediction_of_...,False,True,False,False,NaN,True,False,False,...,False,False,False,True,False,False,False,False,Repository contains a single large R script th...,[r]
7,PMC10658003,https://github.com/IreneSophia/MLSPE,False,True,True,False,NaN,False,True,True,...,True,True,False,False,False,True,True,True,Strengths: clear high-level README describing ...,"[praat, python, r]"
8,PMC9720950,https://github.com/mi-erasmusmc/EmcDementiaMod...,False,True,True,False,NaN,False,False,True,...,False,False,False,None,False,False,False,False,Repo is primarily a wrapper/landing repo: it c...,[r]
9,PMC9653436,https://github.com/Dexin-Chen/Pathomics-analysis,False,False,False,False,NaN,False,False,False,...,False,False,False,True,False,False,False,False,Repository consists of a single R script (no R...,[r]


# Not downloaded by utility

In [6]:
df_pred_notok = df_pred[df_pred["repo_status"] != RepoStatus.OK]

In [7]:
# Should have been captured, but was missed by repo util
df_pred_notok[df_pred_notok["gt_is_empty"] == False][["repo_url", "repo_status", "repo_error"]]

,repo_url,repo_status,repo_error
13,https://doi.org/10.57770/QTYQZS,inaccessible,HTTPSConnectionPool(host='dataverse.lib.nycu.e...


In [8]:
# These are not accessible by humans / not code - it is normal that the repo util skipped them
df_pred_notok[df_pred_notok["gt_is_empty"] == True]

,PMCID,repo_url,gt_is_empty,gt_contains_readme,gt_readme_purpose_and_outputs,gt_contains_requirements,gt_requirements_dependency_versions,gt_contains_license,gt_sufficient_code_documentation,gt_is_modular_and_structured,...,sufficient_code_documentation,is_modular_and_structured,implements_tests,fixes_seed_if_stochastic,lists_hardware_requirements,contains_link_to_paper,contains_citation,includes_data_or_sample,comments_and_explanations,coding_languages
0,PMC11307559,https://seakheeoh76.shinyapps.io/XGBoost_BA_pr...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,PMC8175333,https://github.com/ATARIQ5,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19,PMC11992218,https://www.cdc.gov/nchs/nhanes/index.htm,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
21,PMC10844815,https://github.com/yiwangz/SS_learning/MLinHCT,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
30,PMC8784888,https://github.com/AkihiroShiroshita/Predictio...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


# Performance analysis

In [9]:
df_pred_ok = df_pred[df_pred["repo_status"] == RepoStatus.OK]

In [10]:
df_pred_ok[["repo_url", "contains_link_to_paper", "gt_contains_link_to_paper"]]

,repo_url,contains_link_to_paper,gt_contains_link_to_paper
1,https://github.com/EnzhaoZhu/Common-risks-pred...,False,False
2,https://github.com/CalvinSMU/AiLES,True,True
4,https://github.com/Aneja-Lab-Yale/Aneja-Lab-Pu...,True,True
5,https://github.com/laamit/champCalculator,False,False
6,https://github.com/TheDecodeLab/Prediction_of_...,False,False
7,https://github.com/IreneSophia/MLSPE,True,False
8,https://github.com/mi-erasmusmc/EmcDementiaMod...,False,False
9,https://github.com/Dexin-Chen/Pathomics-analysis,False,False
10,https://doi.org/10.5281/zenodo.4560078,True,False
11,https://github.com/jacksoncoki/Emergency-data-...,False,False


In [ ]:
import numpy as np
import pandas as pd

fields = [
    "contains_readme",
    "readme_purpose_and_outputs",
    "contains_requirements",
    "requirements_dependency_versions",
    "contains_license",
    "sufficient_code_documentation",
    "is_modular_and_structured",
    "implements_tests",
    "fixes_seed_if_stochastic",
    "lists_hardware_requirements",
    "contains_link_to_paper",
    "contains_citation",
    "includes_data_or_sample",
]

metrics = []

for field in fields:
    gt_col = f"gt_{field}"
    pred_col = f"{field}"

    # Drop rows where either GT or prediction is NaN for this field
    sub = df_pred_ok[[gt_col, pred_col]].dropna()

    if sub.empty:
        continue

    y_true = sub[gt_col].astype(bool)
    y_pred = sub[pred_col].astype(bool)

    tp = ((y_true) & (y_pred)).sum()
    tn = ((~y_true) & (~y_pred)).sum()
    fp = ((~y_true) & (y_pred)).sum()
    fn = ((y_true) & (~y_pred)).sum()

    accuracy = (y_true == y_pred).mean()
    recall_true = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision_true = tp / (tp + fp) if (tp + fp) > 0 else np.nan

    f1 = 2 * (precision_true * recall_true) / (precision_true + recall_true) if (precision_true + recall_true) > 0 else np.nan
    
    metrics.append(
        {
            "field": field,
            "accuracy": accuracy,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "recall_true": recall_true,
            "precision_true": precision_true,
            "f1": f1,
            "n_non_null": len(sub),

        }
    )
    metrics_df = pd.DataFrame(metrics).set_index("field")
    display(metrics_df.round(3))
    # Calculate macro-averaged F1 score
    macro_f1 = metrics_df['f1'].mean()
    print(f"\nMacro-averaged F1 score: {macro_f1:.3f}")

    # Calculate weighted F1 score
    weighted_f1 = (metrics_df['f1'] * metrics_df['n_non_null']).sum() / metrics_df['n_non_null'].sum()
    print(f"Weighted F1 score: {weighted_f1:.3f}")

    # Calculate weighted accuracy, recall, and precision
    weighted_accuracy = (metrics_df['accuracy'] * metrics_df['n_non_null']).sum() / metrics_df['n_non_null'].sum()
    weighted_recall = (metrics_df['recall_true'] * metrics_df['n_non_null']).sum() / metrics_df['n_non_null'].sum()
    weighted_precision = (metrics_df['precision_true'] * metrics_df['n_non_null']).sum() / metrics_df['n_non_null'].sum()

    print(f"Weighted Accuracy: {weighted_accuracy:.3f}")
    print(f"Weighted Recall: {weighted_recall:.3f}")
    print(f"Weighted Precision: {weighted_precision:.3f}")


,accuracy,tp,tn,fp,fn,recall_true,precision_true,f1,n_non_null
field,,,,,,,,,
contains_readme,1.000000,24,5,0,0,1.000000,1.000000,1.000000,29
readme_purpose_and_outputs,0.750000,14,4,6,0,1.000000,0.700000,0.823529,24
contains_requirements,0.827586,8,16,5,0,1.000000,0.615385,0.761905,29
requirements_dependency_versions,0.875000,6,1,0,1,0.857143,1.000000,0.923077,8
contains_license,1.000000,8,21,0,0,1.000000,1.000000,1.000000,29
sufficient_code_documentation,0.724138,10,11,4,4,0.714286,0.714286,0.714286,29
is_modular_and_structured,0.896552,17,9,0,3,0.850000,1.000000,0.918919,29
implements_tests,1.000000,1,28,0,0,1.000000,1.000000,1.000000,29
fixes_seed_if_stochastic,0.727273,10,6,4,2,0.833333,0.714286,0.769231,22



Macro-averaged F1 score: 0.8311
Weighted F1 score: 0.8269
Weighted Accuracy: 0.8808
Weighted Recall: 0.9132
Weighted Precision: 0.7959


In [12]:
df_pred.to_csv("../data/03_repo_assessment/evaluation_repo_assessment.csv", index=False)

In [11]:
df_pred_nonempty[["repo_url",
    # "contains_readme",
    # "readme_purpose_and_outputs",
    # "contains_requirements",
    "requirements_dependency_versions",
    "gt_requirements_dependency_versions",
    # "contains_license",
    # "sufficient_code_documentation",
    # "is_modular_and_structured",
    # "implements_tests",
    # "fixes_seed_if_stochastic",
    # "lists_hardware_requirements",
    # "contains_link_to_paper",
    # "contains_citation",
    "gt_includes_data_or_sample",
    "includes_data_or_sample",
]]

,repo_url,requirements_dependency_versions,gt_requirements_dependency_versions,gt_includes_data_or_sample,includes_data_or_sample
1,https://github.com/EnzhaoZhu/Common-risks-pred...,True,True,False,False
4,https://github.com/Aneja-Lab-Yale/Aneja-Lab-Pu...,True,True,False,False
5,https://github.com/laamit/champCalculator,True,True,yes,True
6,https://github.com/TheDecodeLab/Prediction_of_...,None,NaN,False,False
7,https://github.com/IreneSophia/MLSPE,None,NaN,False,True
8,https://github.com/mi-erasmusmc/EmcDementiaMod...,False,NaN,False,False
9,https://github.com/Dexin-Chen/Pathomics-analysis,None,NaN,False,False
10,https://doi.org/10.5281/zenodo.4560078,<NA>,True,True,<NA>
11,https://github.com/jacksoncoki/Emergency-data-...,None,NaN,False,False
12,https://github.com/dt433/Predict-Prostate,None,NaN,False,False


In [19]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

PRED_COLS = [
    "contains_readme",
    "readme_purpose_and_outputs",
    "contains_requirements",
    "requirements_dependency_versions",
    "contains_license",
    "sufficient_code_documentation",
    "is_modular_and_structured",
    "implements_tests",
    "fixes_seed_if_stochastic",
    "lists_hardware_requirements",
    "contains_link_to_paper",
    "contains_citation",
    "includes_data_or_sample",
]

rows = []

for col in PRED_COLS:
    gt_col = f"gt_{col}"
    if col not in df_pred_nonempty.columns or gt_col not in df_pred_nonempty.columns:
        continue

    y_true = df_pred_nonempty[gt_col]
    y_pred = df_pred_nonempty[col]

    m = y_true.notna() & y_pred.notna()
    n = int(m.sum())

    if n == 0:
        rows.append({"field": col, "n": 0, "precision": np.nan, "recall": np.nan, "f1": np.nan})
        continue

    yt = y_true[m].astype(bool).to_numpy()
    yp = y_pred[m].astype(bool).to_numpy()

    p, r, f1, _ = precision_recall_fscore_support(
        yt, yp, average="binary", pos_label=True, zero_division=0
    )

    rows.append({"field": col, "n": n, "precision": p, "recall": r, "f1": f1})

results = pd.DataFrame(rows)
display(results.round(3))

,field,n,precision,recall,f1
0,contains_readme,27,1.000,1.000,1.000
1,readme_purpose_and_outputs,22,0.706,0.923,0.800
2,contains_requirements,27,0.357,1.000,0.526
3,requirements_dependency_versions,14,0.800,0.800,0.800
4,contains_license,27,1.000,1.000,1.000
5,sufficient_code_documentation,27,0.667,0.667,0.667
6,is_modular_and_structured,27,0.933,0.778,0.848
7,implements_tests,27,1.000,1.000,1.000
8,fixes_seed_if_stochastic,20,0.636,0.700,0.667
9,lists_hardware_requirements,27,1.000,1.000,1.000


In [ ]:
# Repos not processed

df_repos_assessment[df_repos_assessment["repo_content"].isna()][["PMCID", "repo_url"]]

In [ ]:
# Count rows where repo_content is not null
n_with_content = df_repos_assessment["repo_content"].notna().sum()
print(f"Number of rows with repo content: {n_with_content}")

In [9]:
df_pred.columns

Index(['PMID', 'repo_url', 'gt_is_empty', 'gt_contains_readme',
       'gt_readme_purpose_and_outputs', 'gt_contains_requirements',
       'gt_requirements_dependency_versions', 'gt_contains_license',
       'gt_sufficient_code_documentation', 'gt_is_modular_and_structured',
       'gt_implements_tests', 'gt_fixes_seed_if_stochastic',
       'gt_lists_hardware_requirements', 'gt_contains_link_to_paper',
       'gt_contains_citation', 'gt_includes_data_or_sample', 'repo_content',
       'repo_status', 'repo_error', 'generation', 'is_empty',
       'contains_readme', 'readme_purpose_and_outputs',
       'contains_requirements', 'requirements_dependency_versions',
       'contains_license', 'sufficient_code_documentation',
       'is_modular_and_structured', 'implements_tests',
       'fixes_seed_if_stochastic', 'lists_hardware_requirements',
       'contains_link_to_paper', 'contains_citation',
       'includes_data_or_sample', 'comments_and_explanations',
       'coding_languages'],
   

In [13]:
import numpy as np
import pandas as pd

fields = [
    "contains_readme",
    "readme_purpose_and_outputs",
    "contains_requirements",
    "requirements_dependency_versions",
    "contains_license",
    "sufficient_code_documentation",
    "is_modular_and_structured",
    "implements_tests",
    "fixes_seed_if_stochastic",
    "lists_hardware_requirements",
    "contains_link_to_paper",
    "contains_citation",
    "includes_data_or_sample",
]

metrics = []

for field in fields:
    gt_col = f"gt_{field}"
    pred_col = f"{field}"

    # Drop rows where either GT or prediction is NaN for this field
    sub = df_pred[[gt_col, pred_col]].dropna()

    if sub.empty:
        continue

    y_true = sub[gt_col].astype(bool)
    y_pred = sub[pred_col].astype(bool)

    tp = ((y_true) & (y_pred)).sum()
    tn = ((~y_true) & (~y_pred)).sum()
    fp = ((~y_true) & (y_pred)).sum()
    fn = ((y_true) & (~y_pred)).sum()

    accuracy = (y_true == y_pred).mean()
    recall_true = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision_true = tp / (tp + fp) if (tp + fp) > 0 else np.nan

    f1 = 2 * (precision_true * recall_true) / (precision_true + recall_true) if (precision_true + recall_true) > 0 else np.nan
    
    metrics.append(
        {
            "field": field,
            "n_non_null": len(sub),
            "accuracy": accuracy,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "recall_true": recall_true,
            "precision_true": precision_true,
            "f1": f1,
        }
    )

metrics_df = pd.DataFrame(metrics).set_index("field")
display(metrics_df)
# Calculate macro-averaged F1 score
macro_f1 = metrics_df['f1'].mean()
print(f"\nMacro-averaged F1 score: {macro_f1:.4f}")

# Calculate weighted F1 score
weighted_f1 = (metrics_df['f1'] * metrics_df['n_non_null']).sum() / metrics_df['n_non_null'].sum()
print(f"Weighted F1 score: {weighted_f1:.4f}")

,n_non_null,accuracy,tp,tn,fp,fn,recall_true,precision_true,f1
field,,,,,,,,,
contains_readme,29,1.000000,24,5,0,0,1.000000,1.000000,1.000000
readme_purpose_and_outputs,24,0.708333,13,4,6,1,0.928571,0.684211,0.787879
contains_requirements,29,0.724138,6,15,8,0,1.000000,0.428571,0.600000
requirements_dependency_versions,6,1.000000,6,0,0,0,1.000000,1.000000,1.000000
contains_license,29,1.000000,8,21,0,0,1.000000,1.000000,1.000000
sufficient_code_documentation,29,0.689655,9,11,4,5,0.642857,0.692308,0.666667
is_modular_and_structured,29,0.862069,16,9,0,4,0.800000,1.000000,0.888889
implements_tests,29,1.000000,1,28,0,0,1.000000,1.000000,1.000000
fixes_seed_if_stochastic,21,0.666667,8,6,3,4,0.666667,0.727273,0.695652



Macro-averaged F1 score: 0.7760
Weighted F1 score: 0.7627


In [ ]:
fp_is_empty = df_pred.loc[
    (df_pred["is_empty_pred"] == True) &
    (df_pred["is_empty_gt"] == False),
    ["PMCID", "repo_url", "is_empty_pred", "is_empty_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

In [ ]:
column_name = "contains_readme"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == False) &
    (df_pred[f"{column_name}_gt"] == True),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

In [ ]:
column_name = "contains_requirements"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == False) &
    (df_pred[f"{column_name}_gt"] == True),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

In [ ]:
column_name = "contains_requirements"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == True) &
    (df_pred[f"{column_name}_gt"] == False),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

In [ ]:
column_name = "requirements_dependency_versions"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == True) &
    (df_pred[f"{column_name}_gt"] == False),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

In [ ]:


column_name = "readme_purpose_and_outputs"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == True) &
    (df_pred[f"{column_name}_gt"] == False),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))